In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# A Parser for Propositional Logic

## Representing Formulas

In TypeScript, formulas are represented as nested arrays.  In OCaml, we use a *variant type*.  A variant type is a type whose values are built from a fixed set of *constructors*.  Every constructor can carry arguments.  The type `formula` given below has one constructor for every kind of formula:

| Formula               | OCaml representation |
|:----------------------|:---------------------|
| $p$                   | `Var "p"`            |
| $\top$                | `Verum`              |
| $\bot$                | `Falsum`             |
| $\neg f$              | `Not f`              |
| $f \wedge g$          | `And (f, g)`         |
| $f \vee g$            | `Or (f, g)`          |
| $f \rightarrow g$     | `Imp (f, g)`         |
| $f \leftrightarrow g$ | `Iff (f, g)`         |

The type is *recursive*: the arguments of the constructors `Not`, `And`, `Or`, `Imp`, and `Iff` are again formulas.  For example, the formula $(p \rightarrow q) \wedge \neg r$ is represented as
```
And (Imp (Var "p", Var "q"), Not (Var "r"))
```
In contrast to nested arrays, the OCaml compiler checks that every value of type `formula` is a well-formed formula.  For example, it is impossible to apply `Not` to two arguments.

In [ ]:
type formula =
  | Var of string               (* propositional variable *)
  | Verum                       (* ⊤ *)
  | Falsum                      (* ⊥ *)
  | Not of formula              (* ¬ *)
  | And of formula * formula    (* ∧ *)
  | Or  of formula * formula    (* ∨ *)
  | Imp of formula * formula    (* → *)
  | Iff of formula * formula    (* ↔ *)

In [ ]:
And (Imp (Var "p", Var "q"), Not (Var "r"))

## The Tokenizer

OCaml strings are sequences of *bytes*.  The logical operators are Unicode characters that are encoded in UTF-8.  In this encoding, they occupy two or three bytes.  For example, the string `"∧"` has length 3.

In [ ]:
String.length "∧"

The list `operators` contains the logical operators that are written with special characters.

In [ ]:
let operators : string list = ["⊤"; "⊥"; "∧"; "∨"; "¬"; "→"; "↔"]

The function `is_letter` takes one argument:
- `c` is a character.

It returns `true` if `c` is a letter from the Latin alphabet.

The function `is_ident_char` takes one argument:
- `c` is a character.

It returns `true` if `c` can occur in an identifier after the first character, i.e. if `c` is a letter, a digit, or one of the characters `<`, `>`, and `,`.  These characters are allowed in variable names such as `Q<1,2>`.

In [ ]:
let is_letter (c : char) : bool =
  ('A' <= c && c <= 'Z') || ('a' <= c && c <= 'z')

let is_ident_char (c : char) : bool =
  is_letter c || ('0' <= c && c <= '9') || c = '<' || c = '>' || c = ','

The function `tokenize` takes one argument:
- `s` is a string that is supposed to represent a formula from propositional logic.

The function returns the list of *tokens* of `s`.  A token is either
- an *identifier*, i.e. a string that starts with a letter that is followed by letters, digits, or one of the characters `<`, `>`, and `,`,
- one of the logical operators `⊤`, `⊥`, `∧`, `∨`, `¬`, `→`, `↔`, or
- a parenthesis.

Blanks and tabs are skipped.  If `s` contains any other character, an exception is raised.

**Implementation:** The TypeScript version uses a *regular expression*.  The OCaml standard library does not provide regular expressions.  Therefore, the string is scanned from left to right by the recursive function `scan`.  The call `scan i` returns the tokens of the part of `s` that starts at position `i`.
- The auxiliary function `ident_end j` returns the position of the first character at or after position `j` that can not be part of an identifier.  Hence, if an identifier starts at position `i`, it ends before position `ident_end (i + 1)`.
- `starts_at op i` checks whether the operator `op` occurs in `s` at position `i`.  As the operators occupy several bytes, the function compares the substring of `s` that starts at `i` and has the length of `op` with `op`.
- `List.find_opt p l` returns `Some x` for the first element `x` of `l` that satisfies the predicate `p`, or `None` if there is no such element.
- `String.make 1 c` turns the character `c` into a string of length 1.

In [ ]:
let tokenize (s : string) : string list =
  let n = String.length s in
  let rec ident_end j =
    if j < n && is_ident_char s.[j] then ident_end (j + 1) else j
  in
  let starts_at op i =
    i + String.length op <= n && String.sub s i (String.length op) = op
  in
  let rec scan i =
    if i >= n then
      []
    else if s.[i] = ' ' || s.[i] = '\t' then
      scan (i + 1)
    else if s.[i] = '(' || s.[i] = ')' then
      String.make 1 s.[i] :: scan (i + 1)
    else if is_letter s.[i] then
      let j = ident_end (i + 1) in
      String.sub s i (j - i) :: scan j
    else
      match List.find_opt (fun op -> starts_at op i) operators with
      | Some op -> op :: scan (i + String.length op)
      | None    -> failwith (Printf.sprintf "tokenize: unexpected character at position %d of %s" i s)
  in
  scan 0

In [ ]:
tokenize "p ↔ q ↔ r"

In [ ]:
tokenize "¬(a<1,2> ∧ b) → ⊥"

The function `is_prop_var` takes one argument:
- `s` is a string.

The function returns `true` if `s` can be interpreted as a propositional variable, i.e. if `s` is an identifier.  `String.for_all p s` checks whether all characters of `s` satisfy the predicate `p`.

In [ ]:
let is_prop_var (s : string) : bool =
  s <> "" && is_letter s.[0] && String.for_all is_ident_char s

In [ ]:
is_prop_var "a<1,2>", is_prop_var "∧"

## The Shunting Yard Algorithm

The parser implements the *shunting yard algorithm* that was invented by Edsger W. Dijkstra.  It uses two stacks:
- `arguments` is a stack of formulas that have already been parsed,
- `operators` is a stack of operators (and opening parentheses) that are still waiting for their arguments.

Before we can define the parser, we need to define the *precedences* of the operators.  The function `precedence` takes one argument:
- `op` is a string representing a logical operator.

It returns the precedence of `op`.  Operators with a higher precedence bind stronger.  For example, as $\wedge$ has a higher precedence than $\vee$, the formula $p \vee q \wedge r$ is parsed as $p \vee (q \wedge r)$.  If `op` is not a logical operator, an exception is raised.

In [ ]:
let precedence (op : string) : int =
  match op with
  | "↔"       -> 1
  | "→"       -> 2
  | "∨"       -> 3
  | "∧"       -> 4
  | "¬"       -> 5
  | "⊤" | "⊥" -> 6
  | _         -> failwith ("precedence: unknown operator " ^ op)

The function `eval_before` takes two arguments:
- `stack_op` is the operator on top of the operator stack,
- `next_op` is the next operator from the input.

The function returns `true` if the operator `stack_op` has to be evaluated before the operator `next_op` is processed.  This is the case if
- `stack_op` has a higher precedence than `next_op`, or
- both operators have the same precedence and they are different, or
- both operators are identical and are *left associative*.  The operators `∧` and `∨` are left associative, while `→` and `↔` are *right associative*.  For example, $p \rightarrow q \rightarrow r$ is parsed as $p \rightarrow (q \rightarrow r)$.

An opening parenthesis on the operator stack is never evaluated: it has to wait for the matching closing parenthesis.

In [ ]:
let eval_before (stack_op : string) (next_op : string) : bool =
  if stack_op = "(" then
    false
  else if precedence stack_op > precedence next_op then
    true
  else if precedence stack_op = precedence next_op then
    if stack_op = next_op then
      stack_op = "∧" || stack_op = "∨"    (* left associative *)
    else
      true
  else
    false

The function `apply` takes two arguments:
- `op` is a string representing a logical operator,
- `arguments` is the stack of formulas, represented as a list where the top of the stack is the head of the list.

The function takes the arguments of `op` from the stack `arguments`, builds the formula that results from applying `op` to these arguments, and pushes this formula onto the stack.  The new stack is returned.
- The operators `⊤` and `⊥` take no arguments.
- The operator `¬` takes one argument.
- All other operators take two arguments.  As the left argument was pushed first, the right argument `rhs` is on top of the stack.

If there are not enough arguments on the stack, the input is not a well-formed formula and an exception is raised.

**Implementation:** The function matches the pair consisting of the operator and the stack.  A pattern like `rhs :: lhs :: rest` only matches a list that has at least two elements.

In [ ]:
let apply (op : string) (arguments : formula list) : formula list =
  match op, arguments with
  | "⊤", _                  -> Verum :: arguments
  | "⊥", _                  -> Falsum :: arguments
  | "¬", f :: rest          -> Not f :: rest
  | "∧", rhs :: lhs :: rest -> And (lhs, rhs) :: rest
  | "∨", rhs :: lhs :: rest -> Or  (lhs, rhs) :: rest
  | "→", rhs :: lhs :: rest -> Imp (lhs, rhs) :: rest
  | "↔", rhs :: lhs :: rest -> Iff (lhs, rhs) :: rest
  | _                       -> failwith ("apply: syntax error, missing argument for " ^ op)

The function `parse` takes one argument:
- `s` is a string that represents a formula from propositional logic.

The function returns the formula represented by `s`, i.e. the *syntax tree* of `s`.  If `s` can not be parsed, an exception is raised.

**Implementation:** The tokens of `s` are processed from left to right by the recursive function `loop`.  It takes three arguments, which are all lists that are used as stacks:
- `tokens` contains the tokens that still have to be processed,
- `operators` is the operator stack,
- `arguments` is the argument stack.

For the next token `next_op` taken from `tokens` we distinguish the following cases:
1. If `next_op` is a propositional variable, the formula `Var next_op` is pushed onto the argument stack.
2. If the operator stack is empty, or if `next_op` is `⊤`, `⊥`, or an opening parenthesis, then `next_op` is pushed onto the operator stack.
3. Otherwise, let `stack_op` be the operator on top of the operator stack.
   - If `stack_op` is an opening parenthesis and `next_op` is a closing parenthesis, the opening parenthesis is removed from the operator stack.
   - If `next_op` is a closing parenthesis, or if `stack_op` has to be evaluated before `next_op`, then `stack_op` is removed from the operator stack and applied to its arguments.  The token `next_op` is not removed from `tokens`, so that it will be processed again.
   - Otherwise, `next_op` is pushed onto the operator stack.

Once all tokens have been processed, the remaining operators on the operator stack are applied.  `List.fold_left` applies them from the top of the stack to its bottom.  If the input was a well-formed formula, the argument stack now contains exactly one formula, which is the result.

Every recursive call of `loop` is the last action of `loop`.  Such a *tail call* does not need any space on the stack, i.e. `loop` is as efficient as a `while` loop.

In [ ]:
let parse (s : string) : formula =
  let rec loop (tokens : string list) (operators : string list) (arguments : formula list) : formula =
    match tokens, operators with
    | [], _ ->
        (match List.fold_left (fun args op -> apply op args) arguments operators with
         | [f] -> f
         | _   -> failwith ("parse: could not parse " ^ s))
    | next_op :: rest, _ when is_prop_var next_op ->
        loop rest operators (Var next_op :: arguments)
    | next_op :: rest, [] ->
        loop rest [next_op] arguments
    | next_op :: rest, _ when next_op = "⊤" || next_op = "⊥" || next_op = "(" ->
        loop rest (next_op :: operators) arguments
    | next_op :: rest, stack_op :: ops ->
        if stack_op = "(" && next_op = ")" then
          loop rest ops arguments
        else if next_op = ")" || eval_before stack_op next_op then
          loop tokens ops (apply stack_op arguments)
        else
          loop rest (next_op :: operators) arguments
  in
  loop (tokenize s) [] []

## Testing

In [ ]:
parse "¬⊥"

In [ ]:
parse "¬p ↔ (p → ⊥)"

In [ ]:
parse "¬⊥ ↔ ⊤"

In [ ]:
parse "p ∧ q"

The operator $\wedge$ binds stronger than $\vee$:

In [ ]:
parse "p ∨ q ∧ r"

In [ ]:
parse "p ∧ q ∨ r"

In [ ]:
parse "p ∧ q → r ∨ s"

The operator $\rightarrow$ is right associative:

In [ ]:
parse "p → q → r"

In [ ]:
parse "p ∧ q ↔ q ∨ p"

In [ ]:
parse "¬(p ∨ q) ↔ ¬p ∨ ¬q"

In [ ]:
parse "a<1,2> ↔ b<2,1>"

Strings that are not well-formed formulas are rejected.  The following cell raises an exception on purpose.

In [ ]:
parse "p ∧ ∨ q"